# Import necessary libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load dataset from CSV file

In [2]:
df = pd.read_csv(r"D:\Experiment\A-Z ML\Nlp\Restrant review\Dataset\Restaurant_Reviews.tsv",delimiter='\t')

In [3]:
# Display the first few rows of the dataset
df.head()

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [4]:
# Count the number of positive and negative reviews
df['Liked'].value_counts()

Liked
1    500
0    500
Name: count, dtype: int64

In [5]:
# Calculate the total number of reviews
len(df)

1000

In [6]:
# Separate features (X) and target (y)
X,y = df.iloc[:,0],df.iloc[:,-1]

# Import libraries for text preprocessing

In [7]:
import re 
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer  #reducing words to their base or root form example, the words "running",
                                            #"runner", and "ran" would be reduced to "run".
from textblob import TextBlob

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Smile\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Perform text preprocessing on each review

In [8]:
corpus = []
for i in range(0,1000):
    
    review = re.sub('[^a-zA-Z]'," ",X.loc[i])
    review = review.lower()
    review = review.split()
    
    ps = PorterStemmer()
    all_stopwords = stopwords.words('english')
    all_stopwords.remove('not')
    review = [ ps.stem(word) for word in review if word not in set(all_stopwords) ]
    review = " ".join(review)
    review = TextBlob(review).correct() # Correct the spelling mistakes
    review = str(review)
    corpus.append(review)

In [9]:
# all_stopwords is a list conatins all stop words from english 

In [10]:
# Display a sample of preprocessed reviews
corpus[:20]

['now love place',
 'crust not good',
 'not taste texture nasty',
 'stop late may bank holiday rich steve recommend love',
 'select menu great price',
 'get angry want damn who',
 'honest last fresh',
 'potato like rubber could tell made ahead time kept warmer',
 'fro great',
 'great touch',
 'service prompt',
 'would not go back',
 'cashier care ever say still end way over',
 'try cape cod revolt chicken cranberri mmm',
 'disgust pretty sure human hair',
 'shock sign india cash',
 'highly recommend',
 'witness little slow service',
 'place not worth time let along vera',
 'not like']

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

In [12]:
# Import CountVectorizer for converting text into numerical features
cv = CountVectorizer()
X = cv.fit_transform(corpus).toarray()

In [13]:
# Import train_test_split for splitting data into training and testing sets
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=2)

In [14]:
# Import necessary models for classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from scipy.stats import uniform, randint

In [15]:
# Define hyperparameter grids for different classifiers
param_dist = {
    'RandomForestClassifier': {
        'n_estimators': [int(x) for x in np.linspace(start=100, stop=1200, num=12)],
        'max_features': [ 'sqrt', 'log2'],
        'max_depth': [int(x) for x in np.linspace(10, 110, num=11)] + [None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    },
    'DecisionTreeClassifier': {
        'max_features': [ 'sqrt', 'log2', None],
        'max_depth': [int(x) for x in np.linspace(10, 110, num=11)] + [None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'criterion': ['gini', 'entropy']
    },
    'LogisticRegression': {
        'penalty': ['l1', 'l2', 'elasticnet', None],
        'C': np.logspace(-4, 4, 20),
        'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
        'max_iter': [100, 200, 300, 400, 500]
    },
    
    'SVM' :  {
    'C': uniform(loc=0.1, scale=10.0),  # Continuous uniform distribution for C
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],  # Kernels to try
    'gamma': ['scale', 'auto'] + list(np.logspace(-3, 3, 7)),  # Gamma options
    'degree': randint(2, 6),  # Random integer between 2 and 5 for polynomial degree
    'coef0': uniform(loc=-1.0, scale=2.0)  # Uniform distribution for coef0
},
    'NavieBayes' : {
    'alpha': uniform(loc=0.1, scale=10.0),  # Continuous uniform distribution for alpha
    'fit_prior': [True, False]  # Whether to learn class prior probabilities or not
}
}

In [16]:
# Define classifiers
models = {
            'RandomForestClassifier' : RandomForestClassifier(),
             'DecisionTreeClassifier' : DecisionTreeClassifier(),
            'LogisticRegression' : LogisticRegression(),
            'SVM' : SVC(),
            'NavieBayes' : MultinomialNB()
            
            
}

In [17]:
# Dictionary to store trained models
trained_model = {}

In [18]:
# Import metrics for evaluation
from sklearn.metrics import accuracy_score,precision_score,recall_score,confusion_matrix

In [19]:
# Train each model with randomized search for hyperparameter tuning
for model_name,parameters in param_dist.items():
    
    print(model_name)
    estimator = models[model_name]
    random_search = RandomizedSearchCV(estimator=estimator,param_distributions=parameters,verbose=2,n_jobs=-1,cv=3)
    random_search.fit(X_train,y_train)
    trained_model[model_name] = random_search.best_estimator_
    print("The beat parametes ",random_search.best_params_)
    print()
    print()
    y_pred = trained_model[model_name].predict(X_test)
    print("The accuracy score is ",accuracy_score(y_test,y_pred)*100)
    print("The precision score is ",precision_score(y_test,y_pred)*100)
    print("The recall score is ",recall_score(y_test,y_pred)*100)
    print()
    print()
    print(confusion_matrix(y_test,y_pred))
    print()
    print()
    

RandomForestClassifier
Fitting 3 folds for each of 10 candidates, totalling 30 fits
The beat parametes  {'n_estimators': 1200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 110, 'bootstrap': True}


The accuracy score is  80.4
The precision score is  80.53097345132744
The recall score is  77.11864406779661


[[110  22]
 [ 27  91]]


DecisionTreeClassifier
Fitting 3 folds for each of 10 candidates, totalling 30 fits
The beat parametes  {'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 80, 'criterion': 'entropy'}


The accuracy score is  76.8
The precision score is  75.86206896551724
The recall score is  74.57627118644068


[[104  28]
 [ 30  88]]


LogisticRegression
Fitting 3 folds for each of 10 candidates, totalling 30 fits
The beat parametes  {'solver': 'newton-cg', 'penalty': 'l2', 'max_iter': 400, 'C': 1.623776739188721}


The accuracy score is  80.4
The precision score is  77.16535433070865
The recall score is

#  Since the accuracy of the SVM is high, choosing it as the final model.

In [20]:
# Select the SVM model as the final model
model = trained_model['SVM']

In [26]:
# Save the final model using joblib
from joblib import dump

dump(model,r"D:\Experiment\A-Z ML\Nlp\Restrant review\Model\SVC.joblib")
dump(cv,r"D:\Experiment\A-Z ML\Nlp\Restrant review\Model\CountVectorizer.joblib")

['D:\\Experiment\\A-Z ML\\Nlp\\Restrant review\\Model\\CountVectorizer.joblib']

In [22]:
# !jupyter nbconvert --to script model.ipynb


## Sample Prediction

In [32]:
input_text = ["this meat is very good"]
dense_input = cv.transform(input_text).toarray()  # Convert sparse to dense

predicted_label = model.predict(dense_input)
print(predicted_label)

[1]


In [33]:
input_text = ["this meat is not very good"]
dense_input = cv.transform(input_text).toarray()  # Convert sparse to dense

predicted_label = model.predict(dense_input)
print(predicted_label)

[0]
